In [ ]:
# =========================
# Cell 1: 导入库与路径设置
# =========================

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from pathlib import Path

BASE = Path("/Users/qinjiayi/Desktop/ds in empirical study/")  # 改成你的真实路径

YEAR_PATH = BASE / "firm_year_panel(2).xlsx"
QUARTER_PATH = BASE / "firm_quarter_panel（2）.csv"

In [ ]:
# =========================
# Cell 2: 读取季度面板与年度面板
# =========================

df_q = pd.read_csv(QUARTER_PATH, dtype={"gvkey": str})
df_y = pd.read_excel(YEAR_PATH, dtype={"gvkey": str})

print("Quarter panel:", df_q.shape)
print("Year panel:", df_y.shape)

print("\nQuarter years:", sorted(df_q["year"].dropna().unique()))
print("Year panel years:", sorted(df_y["year"].dropna().unique()))

In [ ]:
print(df_q.columns.tolist())
print(df_q[["gvkey", "year"]].head())

In [ ]:
# =========================
# Cell 3: 检查关键变量
# =========================

required_y_vars = [
    "gvkey", "year", "gsector",
    "rd", "capex",
    "rd_future_1y", "capex_future_1y",
    "size", "leverage", "bm", "profitability", "investment",
    "bert_adoption_log_count",
    "bert_innovation_log_count",
    "bert_hype_log_count",
    "bert_risk_log_count",
    "bert_adoption_count",
    "bert_adoption_dummy",
    "bert_adoption_per_1k_words"
]

missing_y = [v for v in required_y_vars if v not in df_y.columns]

print("Missing variables in annual panel:")
print(missing_y if missing_y else "No missing variables. Good.")

print("\nAnnual panel columns preview:")
print(df_y.columns.tolist()[:80])

In [ ]:
# =========================
# Cell 3A: 在年度面板中补造新版 narrative 变量
# =========================

# 1. log_count 变量
for label in ["adoption", "innovation", "hype", "risk"]:
    count_col = f"bert_{label}_count"
    log_col = f"bert_{label}_log_count"
    
    if count_col in df_y.columns and log_col not in df_y.columns:
        df_y[log_col] = np.log1p(df_y[count_col].fillna(0))
        print(f"Created {log_col} from {count_col}")

# 2. per_1k_words 变量
for label in ["adoption", "innovation", "hype", "risk"]:
    count_col = f"bert_{label}_count"
    per1k_col = f"bert_{label}_per_1k_words"
    
    if count_col in df_y.columns and per1k_col not in df_y.columns:
        df_y[per1k_col] = np.where(
            df_y["total_word_count"].fillna(0) > 0,
            df_y[count_col].fillna(0) / df_y["total_word_count"] * 1000,
            np.nan
        )
        print(f"Created {per1k_col} from {count_col} / total_word_count")

# 3. 再次检查
required_y_vars = [
    "bert_adoption_log_count",
    "bert_innovation_log_count",
    "bert_hype_log_count",
    "bert_risk_log_count",
    "bert_adoption_per_1k_words"
]

missing_after_create = [v for v in required_y_vars if v not in df_y.columns]

print("\nMissing after creation:")
print(missing_after_create if missing_after_create else "No missing variables. Good.")

In [ ]:
# =========================
# Cell 4: 构造年度 validation 样本
# =========================

work_y = df_y.copy()

# 基础清洗
work_y = work_y.replace([np.inf, -np.inf], np.nan)

# 只保留有 gvkey 和年份的年度样本
work_y = work_y.dropna(subset=["gvkey", "year"]).copy()

# 年份转整数，方便 FE
work_y["year"] = work_y["year"].astype(int)

print("Annual validation base sample:", work_y.shape)
print("Number of firms:", work_y["gvkey"].nunique())
print("Years:", sorted(work_y["year"].unique()))

In [ ]:
# =========================
# Cell 5: 年度 validation 样本缺失率检查
# =========================

check_vars = [
    "rd_future_1y", "capex_future_1y",
    "bert_adoption_log_count",
    "bert_innovation_log_count",
    "bert_hype_log_count",
    "bert_risk_log_count",
    "size", "leverage", "bm", "profitability", "investment",
    "rd", "capex", "gsector"
]

missing_summary = work_y[check_vars].isna().mean().sort_values(ascending=False)
display(missing_summary.to_frame("missing_rate"))

print("R&D regression available sample:")
print(work_y.dropna(subset=[
    "rd_future_1y", "bert_adoption_log_count",
    "size", "leverage", "bm", "profitability", "investment", "rd", "gsector"
]).shape)

print("\nCapex regression available sample:")
print(work_y.dropna(subset=[
    "capex_future_1y", "bert_adoption_log_count",
    "size", "leverage", "bm", "profitability", "investment", "capex", "gsector"
]).shape)

In [ ]:
# =========================
# Cell 6: 不再手动生成季度 lead
# =========================

# 注意：
# 旧版 notebook 用 groupby("gvkey").shift(-1) 构造 rd_lead1 / capex_lead1。
# 现在不再使用该逻辑。
# 因为 R&D / capex 是年度 Compustat 变量，必须使用年度面板中的：
# rd_future_1y, capex_future_1y。

print("Use rd_future_1y and capex_future_1y from annual panel.")

In [ ]:
# =========================
# Cell 7: winsorize 函数
# =========================

def winsorize_series(s, lower=0.01, upper=0.99):
    lo = s.quantile(lower)
    hi = s.quantile(upper)
    return s.clip(lo, hi)

winsor_vars = [
    "rd_future_1y", "capex_future_1y",
    "rd", "capex",
    "size", "leverage", "bm", "profitability", "investment"
]

for v in winsor_vars:
    if v in work_y.columns:
        work_y[v + "_w"] = winsorize_series(work_y[v])

In [ ]:
# =========================
# Cell 8: 年度 validation 样本描述性统计
# =========================

desc_cols = [
    "bert_adoption_log_count",
    "bert_adoption_count",
    "bert_adoption_dummy",
    "bert_adoption_per_1k_words",
    "bert_innovation_log_count",
    "bert_hype_log_count",
    "bert_risk_log_count",
    "rd_future_1y",
    "capex_future_1y",
    "size", "leverage", "bm", "profitability", "investment"
]

desc_cols = [c for c in desc_cols if c in work_y.columns]

display(
    work_y[desc_cols].describe(
        percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
    ).T
)

zero_share = (work_y[desc_cols] == 0).mean().sort_values(ascending=False)
display(zero_share.to_frame("zero_share"))

In [ ]:
# =========================
# Cell 9: AI narrative 分年份、分行业
# =========================

year_summary = work_y.groupby("year")[
    ["bert_adoption_log_count", "bert_innovation_log_count", "bert_hype_log_count"]
].mean()

display(year_summary)

industry_summary = work_y.groupby("gsector")[
    ["bert_adoption_log_count", "bert_innovation_log_count", "bert_hype_log_count"]
].mean().sort_values("bert_adoption_log_count", ascending=False)

display(industry_summary)

In [ ]:
# =========================
# Cell 10: 高低 AI narrative 组比较
# =========================

x = "bert_adoption_log_count"

temp = work_y.dropna(subset=[x]).copy()

# 因为 AI narrative 很稀疏，建议用 dummy-style high group
temp["ai_high"] = (temp[x] > 0).astype(int)

compare_cols = [
    "rd_future_1y", "capex_future_1y",
    "rd", "capex",
    "size", "leverage", "bm", "profitability", "investment"
]

compare_cols = [c for c in compare_cols if c in temp.columns]

group_compare = temp.groupby("ai_high")[compare_cols].mean().T
group_compare["diff_high_minus_low"] = group_compare[1] - group_compare[0]

display(group_compare)

In [ ]:
# =========================
# Cell 11: 设置新版 validation 变量
# =========================

MAIN_AI_VAR = "bert_adoption_log_count"

ROBUST_AI_VARS = [
    "bert_adoption_count",
    "bert_adoption_dummy",
    "bert_adoption_per_1k_words",
    "bert_innovation_log_count",
    "bert_hype_log_count",
    "bert_risk_log_count"
]

ROBUST_AI_VARS = [v for v in ROBUST_AI_VARS if v in work_y.columns]

print("Main AI variable:", MAIN_AI_VAR)
print("Robustness variables:", ROBUST_AI_VARS)

In [ ]:
# =========================
# Cell 12: 年度 validation 回归函数
# =========================

def run_annual_validation(y_var, x_var, data, add_current_y=True):
    controls = ["size", "leverage", "bm", "profitability", "investment"]
    
    if add_current_y:
        if y_var == "rd_future_1y":
            controls.append("rd")
        elif y_var == "capex_future_1y":
            controls.append("capex")
    
    needed = [y_var, x_var, "gvkey", "year", "gsector"] + controls
    reg_data = data.dropna(subset=needed).copy()
    
    formula = (
        f"{y_var} ~ {x_var} + "
        + " + ".join(controls)
        + " + C(gsector) + C(year)"
    )
    
    model = smf.ols(formula, data=reg_data).fit(
        cov_type="cluster",
        cov_kwds={"groups": reg_data["gvkey"]}
    )
    
    out = {
        "outcome": y_var,
        "x_var": x_var,
        "nobs": int(model.nobs),
        "coef": model.params.get(x_var, np.nan),
        "t": model.tvalues.get(x_var, np.nan),
        "p": model.pvalues.get(x_var, np.nan),
        "r2": model.rsquared
    }
    
    return model, out

In [ ]:
# =========================
# Cell 13: 主 validation 回归
# =========================

model_rd_main, rd_main_out = run_annual_validation(
    y_var="rd_future_1y",
    x_var=MAIN_AI_VAR,
    data=work_y,
    add_current_y=True
)

model_capex_main, capex_main_out = run_annual_validation(
    y_var="capex_future_1y",
    x_var=MAIN_AI_VAR,
    data=work_y,
    add_current_y=True
)

main_validation_table = pd.DataFrame([rd_main_out, capex_main_out])
display(main_validation_table)

print(model_rd_main.summary())
print(model_capex_main.summary())

In [ ]:
# =========================
# Cell 14: Narrative subtype validation
# =========================

subtype_vars = [
    "bert_adoption_log_count",
    "bert_innovation_log_count",
    "bert_hype_log_count",
    "bert_risk_log_count"
]

subtype_vars = [v for v in subtype_vars if v in work_y.columns]

rows = []

for x in subtype_vars:
    for y in ["rd_future_1y", "capex_future_1y"]:
        try:
            model, out = run_annual_validation(
                y_var=y,
                x_var=x,
                data=work_y,
                add_current_y=True
            )
            rows.append(out)
        except Exception as e:
            rows.append({
                "outcome": y,
                "x_var": x,
                "error": str(e)
            })

subtype_validation_table = pd.DataFrame(rows)
display(subtype_validation_table)

In [ ]:
# =========================
# Cell 15: Adoption 变量形式 robustness
# =========================

adoption_forms = [
    "bert_adoption_log_count",
    "bert_adoption_count",
    "bert_adoption_dummy",
    "bert_adoption_per_1k_words"
]

adoption_forms = [v for v in adoption_forms if v in work_y.columns]

rows = []

for x in adoption_forms:
    for y in ["rd_future_1y", "capex_future_1y"]:
        try:
            model, out = run_annual_validation(
                y_var=y,
                x_var=x,
                data=work_y,
                add_current_y=True
            )
            rows.append(out)
        except Exception as e:
            rows.append({
                "outcome": y,
                "x_var": x,
                "error": str(e)
            })

adoption_form_table = pd.DataFrame(rows)
display(adoption_form_table)

In [ ]:
# =========================================================
# Cell 15B: Winsorized robustness (年度面板稳健性检验)
# =========================================================
# 目的：检查 AI 对未来 R&D/Capex 的预测力是否受极端值驱动
# 注意：只对 outcome 和财务控制变量进行缩尾，不对 AI narrative 计数变量缩尾

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# 1. 确保缩尾函数已定义 (如果没有定义，请取消下面几行的注释)
# from scipy.stats.mstats import winsorize
# def winsorize_series(s):
#     return pd.Series(winsorize(s, limits=[0.01, 0.01]), index=s.index)

# 2. 准备变量并执行缩尾处理
winsor_base_vars = [
    "rd_future_1y", "capex_future_1y",
    "rd", "capex",
    "size", "leverage", "bm", "profitability", "investment"
]

# 假设你的年度面板变量名是 df_y 或 work_y，这里统一指向 work_y
if 'df_y' in globals() and 'work_y' not in globals():
    work_y = df_y.copy()

for v in winsor_base_vars:
    if v in work_y.columns:
        # 构造带有 _w 后缀的缩尾变量
        work_y[f"{v}_w"] = winsorize_series(work_y[v])

# 3. 内部定义回归函数
def run_annual_validation_winsorized(y_var, x_var, data, add_current_y=True):
    y_w = f"{y_var}_w"
    control_map = {
        "size": "size_w",
        "leverage": "leverage_w",
        "bm": "bm_w",
        "profitability": "profitability_w",
        "investment": "investment_w"
    }
    controls = [v for v in control_map.values() if v in data.columns]
    
    if add_current_y:
        if y_var == "rd_future_1y" and "rd_w" in data.columns:
            controls.append("rd_w")
        elif y_var == "capex_future_1y" and "capex_w" in data.columns:
            controls.append("capex_w")

    needed = [y_w, x_var, "gvkey", "year", "gsector"] + controls
    reg_data = data.dropna(subset=needed).copy()

    formula = f"{y_w} ~ {x_var} + " + " + ".join(controls) + " + C(gsector) + C(year)"
    
    model = smf.ols(formula, data=reg_data).fit(
        cov_type="cluster",
        cov_kwds={"groups": reg_data["gvkey"]}
    )

    return {
        "outcome": y_var,
        "x_var": x_var,
        "spec": "winsorized",
        "nobs": int(model.nobs),
        "coef": model.params.get(x_var, np.nan),
        "t": model.tvalues.get(x_var, np.nan),
        "p": model.pvalues.get(x_var, np.nan),
        "r2": model.rsquared
    }

# 4. 执行回归并生成 winsorized_robustness_table
winsor_x_vars = [
    "bert_adoption_log_count", "bert_innovation_log_count", 
    "bert_hype_log_count", "bert_risk_log_count"
]
winsor_x_vars = [v for v in winsor_x_vars if v in work_y.columns]

winsor_results = []
for x in winsor_x_vars:
    for y in ["rd_future_1y", "capex_future_1y"]:
        try:
            res = run_annual_validation_winsorized(y, x, work_y)
            winsor_results.append(res)
        except Exception as e:
            print(f"Error in {x}-{y}: {e}")

# --- 最终定义表格变量 ---
winsorized_robustness_table = pd.DataFrame(winsor_results)

# 5. 展示结果
if not winsorized_robustness_table.empty:
    # 增加显著性标注
    def get_stars(p):
        if p < 0.01: return "***"
        if p < 0.05: return "**"
        if p < 0.1: return "*"
        return ""
    
    winsorized_robustness_table['sig'] = winsorized_robustness_table['p'].apply(get_stars)
    
    print("✅ Winsorized Robustness Table 已生成:")
    display(winsorized_robustness_table[['outcome', 'x_var', 'coef', 't', 'sig', 'p', 'nobs']])
else:
    print("❌ 未能生成结果，请检查变量是否存在。")

In [ ]:

# =========================
# Cell 15C: Non-BERT robustness
# =========================
# 目的：
# 检查主结果是否只依赖 BERT 分类器。
# 如果 ai_semantic_score / actionable / speculative / LDA topics 得到方向一致或有解释力，
# 说明 AI narrative 的信息含量不是单一 BERT pipeline 的产物。

non_bert_vars = [
    "ai_semantic_score",
    "ai_actionable_share",
    "ai_speculative_share",
    "ai_sentence_ratio",
    "lda_topic2_share",   # automation / process / health care
    "lda_topic8_share",   # machine learning core
    "lda_topic0_share"    # AI technology application
]

non_bert_vars = [v for v in non_bert_vars if v in work_y.columns]

print("Non-BERT robustness variables:")
print(non_bert_vars)

rows = []

for x in non_bert_vars:
    for y in ["rd_future_1y", "capex_future_1y"]:
        try:
            model, out = run_annual_validation(
                y_var=y,
                x_var=x,
                data=work_y,
                add_current_y=True
            )
            out["measure_family"] = "non_BERT"
            rows.append(out)
        except Exception as e:
            rows.append({
                "outcome": y,
                "x_var": x,
                "measure_family": "non_BERT",
                "error": str(e)
            })

non_bert_robustness_table = pd.DataFrame(rows)
display(non_bert_robustness_table)


In [ ]:
# =========================
# Cell 16: annual mean / sum / max / dummy 辅助比较
# =========================

base_var_q = "bert_adoption_log_count"

annual_narrative_compare = (
    df_q
    .groupby(["gvkey", "year"])
    .agg(
        ai_annual_mean=(base_var_q, "mean"),
        ai_annual_sum=(base_var_q, "sum"),
        ai_annual_max=(base_var_q, "max"),
        ai_annual_dummy=(base_var_q, lambda x: int((x > 0).any()))
    )
    .reset_index()
)

df_y_compare = work_y.merge(
    annual_narrative_compare,
    on=["gvkey", "year"],
    how="inner"
)

compare_vars = [
    "ai_annual_mean",
    "ai_annual_sum",
    "ai_annual_max",
    "ai_annual_dummy"
]

display(
    df_y_compare[compare_vars].describe(
        percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
    ).T
)

rows = []

for x in compare_vars:
    for y in ["rd_future_1y", "capex_future_1y"]:
        try:
            model, out = run_annual_validation(
                y_var=y,
                x_var=x,
                data=df_y_compare,
                add_current_y=True
            )
            rows.append(out)
        except Exception as e:
            rows.append({
                "outcome": y,
                "x_var": x,
                "error": str(e)
            })

annual_measure_table = pd.DataFrame(rows)
display(annual_measure_table)

In [ ]:
# =========================
# Cell 17: firm FE 诊断
# =========================

diag_var = "bert_adoption_log_count"

within_std = work_y.groupby("gvkey")[diag_var].std()

print("Number of firms:", within_std.shape[0])
print("Firms with zero within-firm std:", (within_std == 0).sum())
print("Share with zero within-firm std:", (within_std == 0).mean())

display(within_std.describe())

In [ ]:
# =========================
# Cell 18: Size heterogeneity
# =========================

temp = work_y.dropna(subset=["size"]).copy()
temp["large_firm"] = (temp["size"] >= temp["size"].median()).astype(int)

rows = []

for group_value, group_name in [(0, "Small firms"), (1, "Large firms")]:
    sub = temp[temp["large_firm"] == group_value].copy()
    
    for y in ["rd_future_1y", "capex_future_1y"]:
        try:
            model, out = run_annual_validation(
                y_var=y,
                x_var="bert_adoption_log_count",
                data=sub,
                add_current_y=True
            )
            out["group"] = group_name
            rows.append(out)
        except Exception as e:
            rows.append({
                "group": group_name,
                "outcome": y,
                "error": str(e)
            })

size_heterogeneity_table = pd.DataFrame(rows)
display(size_heterogeneity_table)

In [ ]:

# =========================
# Cell 19: Initial R&D heterogeneity - corrected version
# =========================
# 旧版问题：
# rd 的中位数很可能等于 0，直接用 rd >= median 会把几乎所有样本放进 High current R&D，
# 低 R&D 组回归容易报错或没有实际意义。
#
# 新版做法：
# 先按 current R&D 是否为正分组：
#   Group 1: Zero current R&D
#   Group 2: Positive current R&D
# 这更符合数据分布，也避免 median=0 导致的分组失败。

def run_annual_validation_subgroup(y_var, x_var, data, group_name, min_n=200, min_firms=50):
    """
    Subgroup regression with safeguards:
    - Skip groups with too few observations/firms
    - Only include current y control if it has variation inside subgroup
    - Only include FE when there is variation
    """
    controls = ["size", "leverage", "bm", "profitability", "investment"]

    if y_var == "rd_future_1y" and "rd" in data.columns and data["rd"].nunique(dropna=True) > 1:
        controls.append("rd")
    elif y_var == "capex_future_1y" and "capex" in data.columns and data["capex"].nunique(dropna=True) > 1:
        controls.append("capex")

    needed = [y_var, x_var, "gvkey", "year"] + controls
    if "gsector" in data.columns:
        needed.append("gsector")

    reg_data = data.dropna(subset=needed).copy()

    if reg_data.shape[0] < min_n or reg_data["gvkey"].nunique() < min_firms:
        return {
            "group": group_name,
            "outcome": y_var,
            "x_var": x_var,
            "nobs": reg_data.shape[0],
            "n_firms": reg_data["gvkey"].nunique() if "gvkey" in reg_data.columns else np.nan,
            "note": "Skipped: too few observations or firms"
        }

    rhs = [x_var] + controls

    if "gsector" in reg_data.columns and reg_data["gsector"].nunique(dropna=True) > 1:
        rhs.append("C(gsector)")
    if reg_data["year"].nunique(dropna=True) > 1:
        rhs.append("C(year)")

    formula = f"{y_var} ~ " + " + ".join(rhs)

    try:
        model = smf.ols(formula, data=reg_data).fit(
            cov_type="cluster",
            cov_kwds={"groups": reg_data["gvkey"]}
        )
        return {
            "group": group_name,
            "outcome": y_var,
            "x_var": x_var,
            "nobs": int(model.nobs),
            "n_firms": reg_data["gvkey"].nunique(),
            "coef": model.params.get(x_var, np.nan),
            "t": model.tvalues.get(x_var, np.nan),
            "p": model.pvalues.get(x_var, np.nan),
            "r2": model.rsquared,
            "note": ""
        }
    except Exception as e:
        return {
            "group": group_name,
            "outcome": y_var,
            "x_var": x_var,
            "nobs": reg_data.shape[0],
            "n_firms": reg_data["gvkey"].nunique(),
            "error": str(e)
        }

temp = work_y.dropna(subset=["rd"]).copy()
temp["rd_positive_group"] = np.where(temp["rd"] > 0, "Positive current R&D", "Zero current R&D")

rows = []

for group_name, sub in temp.groupby("rd_positive_group"):
    for y in ["rd_future_1y", "capex_future_1y"]:
        rows.append(
            run_annual_validation_subgroup(
                y_var=y,
                x_var="bert_adoption_log_count",
                data=sub,
                group_name=group_name
            )
        )

initial_rd_heterogeneity_table = pd.DataFrame(rows)
display(initial_rd_heterogeneity_table)

# Optional: 在 positive R&D 公司内部再分 low/high positive R&D
positive = temp[temp["rd"] > 0].copy()

if positive.shape[0] > 0:
    positive["positive_rd_high"] = np.where(
        positive["rd"] >= positive["rd"].median(),
        "High positive R&D",
        "Low positive R&D"
    )

    rows_pos = []
    for group_name, sub in positive.groupby("positive_rd_high"):
        for y in ["rd_future_1y", "capex_future_1y"]:
            rows_pos.append(
                run_annual_validation_subgroup(
                    y_var=y,
                    x_var="bert_adoption_log_count",
                    data=sub,
                    group_name=group_name
                )
            )

    positive_rd_heterogeneity_table = pd.DataFrame(rows_pos)
    display(positive_rd_heterogeneity_table)
else:
    positive_rd_heterogeneity_table = pd.DataFrame()


In [ ]:
# =========================
# Cell 20: Industry heterogeneity
# =========================

rows = []

for sector, sub in work_y.groupby("gsector"):
    if sub["gvkey"].nunique() < 30:
        continue
    
    for y in ["rd_future_1y", "capex_future_1y"]:
        try:
            # 行业内不再加 C(gsector)
            controls = ["size", "leverage", "bm", "profitability", "investment"]
            controls += ["rd"] if y == "rd_future_1y" else ["capex"]
            
            needed = [y, "bert_adoption_log_count", "gvkey", "year"] + controls
            reg_data = sub.dropna(subset=needed).copy()
            
            formula = (
                f"{y} ~ bert_adoption_log_count + "
                + " + ".join(controls)
                + " + C(year)"
            )
            
            model = smf.ols(formula, data=reg_data).fit(
                cov_type="cluster",
                cov_kwds={"groups": reg_data["gvkey"]}
            )
            
            rows.append({
                "gsector": sector,
                "outcome": y,
                "nobs": int(model.nobs),
                "coef": model.params.get("bert_adoption_log_count", np.nan),
                "t": model.tvalues.get("bert_adoption_log_count", np.nan),
                "p": model.pvalues.get("bert_adoption_log_count", np.nan),
                "r2": model.rsquared
            })
            
        except Exception as e:
            rows.append({
                "gsector": sector,
                "outcome": y,
                "error": str(e)
            })

industry_heterogeneity_table = pd.DataFrame(rows)
display(industry_heterogeneity_table)


## Result interpretation guide

Use this notebook in the following order:

1. **Main validation**: whether `bert_adoption_log_count` predicts `rd_future_1y` and `capex_future_1y`.
2. **Subtype validation**: whether adoption / innovation / hype / risk narratives carry different real-side information.
3. **Adoption form robustness**: whether results survive replacing `log_count` with `count`, `dummy`, and `per_1k_words`.
4. **Winsorized robustness**: whether results survive trimming extreme values in R&D/capex and financial controls.
5. **Non-BERT robustness**: whether conclusions depend only on BERT labels, or are also supported by semantic, dictionary, and LDA measures.
6. **Heterogeneity**: whether effects differ by firm size, current R&D status, and industry.

Main reporting logic:
- If raw and winsorized results have the same sign and similar significance, the result is not mainly driven by outliers.
- If BERT adoption is significant but hype is not, this supports the idea that not all AI narratives contain the same real-side information.
- If non-BERT measures partly confirm the direction, the narrative signal is more credible.
- If non-BERT measures do not confirm the direction, report BERT results as suggestive and exploratory rather than definitive.


In [ ]:

# =========================
# Final Cell: validation 结果汇总与保存
# =========================

print("Main validation:")
display(main_validation_table)

print("\nSubtype validation:")
display(subtype_validation_table)

print("\nAdoption variable form robustness:")
display(adoption_form_table)

print("\nWinsorized robustness:")
display(winsorized_robustness_table)
print("\nRaw vs. winsorized comparison:")
display(compare_winsor)

print("\nNon-BERT robustness:")
display(non_bert_robustness_table)

print("\nAnnual aggregation measure comparison:")
display(annual_measure_table)

print("\nSize heterogeneity:")
display(size_heterogeneity_table)

print("\nInitial R&D heterogeneity:")
display(initial_rd_heterogeneity_table)

if "positive_rd_heterogeneity_table" in globals() and not positive_rd_heterogeneity_table.empty:
    print("\nPositive R&D subsample heterogeneity:")
    display(positive_rd_heterogeneity_table)

print("\nIndustry heterogeneity:")
display(industry_heterogeneity_table)

# 保存结果
OUT_DIR = BASE / "outputs/tables"
OUT_DIR.mkdir(parents=True, exist_ok=True)

main_validation_table.to_csv(OUT_DIR / "validation_main_results.csv", index=False)
subtype_validation_table.to_csv(OUT_DIR / "validation_subtype_results.csv", index=False)
adoption_form_table.to_csv(OUT_DIR / "validation_adoption_form_robustness.csv", index=False)
winsorized_robustness_table.to_csv(OUT_DIR / "validation_winsorized_robustness.csv", index=False)
compare_winsor.to_csv(OUT_DIR / "validation_raw_vs_winsorized_comparison.csv", index=False)
non_bert_robustness_table.to_csv(OUT_DIR / "validation_non_bert_robustness.csv", index=False)
annual_measure_table.to_csv(OUT_DIR / "validation_annual_measure_comparison.csv", index=False)
size_heterogeneity_table.to_csv(OUT_DIR / "validation_size_heterogeneity.csv", index=False)
initial_rd_heterogeneity_table.to_csv(OUT_DIR / "validation_initial_rd_heterogeneity.csv", index=False)

if "positive_rd_heterogeneity_table" in globals() and not positive_rd_heterogeneity_table.empty:
    positive_rd_heterogeneity_table.to_csv(OUT_DIR / "validation_positive_rd_heterogeneity.csv", index=False)

industry_heterogeneity_table.to_csv(OUT_DIR / "validation_industry_heterogeneity.csv", index=False)

print("Saved all validation result tables.")
